# OneWeb: custom synthetic orbits and satellites

This is an independent hypothetical scenario. Edit the altitude, inclination, plane count, satellites per plane, RAAN span, and Walker F in `orbit_info/custom_shells.json`. Counts specify additional synthetic satellites; real satellites are not subtracted, and approval is not implied. Approved parameters are available, so prefer `*_approved_missing.ipynb` for that scenario. Use this branch for density experiments or when official information is unavailable.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

# Locate the shared module from either the repository or notebook directory.
ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents]
             for p in [base, base / "generate satellite"]
             if (p / "satellite_pipeline.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the WorldNTN repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from satellite_pipeline import (
    download_snapshot, initialize_scenario, read_scenario, load_snapshot,
    export_tracks, preview_export, merge_exports, stamp,
)
# If dependencies are missing, run this in a separate notebook cell:
# %pip install -r "{ROOT / 'requirements.txt'}"
print("Workspace:", ROOT)


In [ ]:
from synthetic_orbits import approved_shells, custom_shells, build_virtual
CONSTELLATION = "oneweb"
MODE = "custom"
SEED = 20261005
scenario, times = read_scenario(ROOT)
print("Common epoch:", times[0])
print("Mode:", MODE, "Seed:", SEED)


In [ ]:
shells = custom_shells(CONSTELLATION, ROOT)
display(pd.DataFrame(shells))


## Generate and archive orbital parameters

Record official parameters and assumed parameters separately. A fixed seed reproduces the RAAN origin, phases, and sampling from the gap template. Use circular SGP4 mean orbits with zero drag; formation control, orbit raising, and collision avoidance are outside this model. RAAN and phase values are neither operator measurements nor known vacant orbital slots.


In [ ]:
real_catalog = None
virtual, orbit_meta = build_virtual(
    CONSTELLATION, MODE, scenario["start_time_utc"], records=real_catalog,
    seed=SEED, root=ROOT, max_age_days=scenario["max_element_age_days"],
)
print("Generated synthetic satellites:", len(virtual))
orbit_dir = ROOT / orbit_meta["orbit_directory"]
display(pd.read_csv(orbit_dir / "population.csv"))
print("Saved mean elements:", orbit_dir / "synthetic_mean_elements.csv")


## Propagate with the shared timeline and CSV schema

Save real orbits, simulations using approved shells, and custom simulations separately. A zero population gap produces an empty CSV with headers and all-zero counts; this is a valid result.


In [ ]:
result = export_tracks(
    virtual, CONSTELLATION, source_type=orbit_meta["assumptions"]["source_type"],
    root=ROOT, extra_metadata=orbit_meta,
)
display(preview_export(result))


## Merge exports later

Merging is disabled by default. Select a real CSV generated with the same `scenario.json` and snapshot before merging. The `approved_missing` and `custom` notebooks represent alternative scenarios for a constellation and must not both be counted in that constellation. The merge function checks the timeline, observer, filters, input hashes, and duplicate IDs.


In [ ]:
REAL_CSV = None  # Set an explicit path to the matching real visibility CSV.
if REAL_CSV is not None:
    merged = merge_exports(
        [REAL_CSV, result["visible_csv"]],
        ROOT / "outputs" / "merged" / f"{CONSTELLATION}_{MODE}_{stamp()}.csv",
    )
    print(merged)
